# 2D Linear Schrödinger Equation: Young's Double Slit Experiment
## Huygens-Fresnel Principle via Source Terms

This notebook simulates wave diffraction and interference using the linear Schrödinger equation. Instead of using a physical barrier and boundary conditions, we model the wall and the two slits using a **spatially localized, oscillating source function** $S(x,y,t)$.

This acts as two coherent emitters (Huygens' wavelets), producing a beautiful interference pattern as the waves propagate through the "vacuum".

## 1. The Governing Equation

$$
i\partial_t\psi + \nabla^2\psi = 0 \quad \implies \quad \partial_t\psi = i\nabla^2\psi + S(x,y,t)
$$

*   **Linear Diffraction:** $i\nabla^2\psi$ governs the dispersive propagation of the wave packet.
*   **Source Term:** $S(x,y,t)$ represents the two slits. It is non-zero only at the slit locations and oscillates at a carrier frequency $\omega_0$ to simulate a monochromatic light source or coherent matter wave.

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Grid and Time ──
# Large domain to prevent boundary reflections during the simulation
Lx, Ly = 100.0, 100.0
Nx, Ny = 512, 512
# Nx, Ny = 128, 128

Lt, Nt = 10.0, 1000
dt = Lt / Nt
n_frames = 150

# ── Slit Geometry & Source ──
x0 = 0.0     # Wall position (emitters location)
d = 5.0       # Slit separation
wx = 1.0       # Wall "thickness" (Gaussian width in x)
wy = 1.0       # Slit width (Gaussian width in y)
omega0 = 5.0   # Carrier frequency of the emitted wave

# ── Source Amplitude Compensation ──
# The solver's default time-stepping adds the source term directly without 
# multiplying by dt (u_new = u_lin + u_nl + source). This effectively scales 
# the physical source by 1/dt. We compensate by scaling our physical amplitude by dt.
S0_phys = 50.0
S0 = S0_phys * dt 

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
psi_func  = sp.Function('psi')
psi_field = psi_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂ψ/∂t = i(∂²ψ/∂x² + ∂²ψ/∂y²) + ...
# Fourier: ∂²/∂x² → -ξ²,  ∂²/∂y² → -η²
# So: i(-ξ² - η²) = -i(ξ² + η²)

symbol_linear = -sp.I * (xi**2 + eta**2)

print("Principal symbol (linear part):")
print("  a(ξ, η) =", symbol_linear)

## 4. Source term and Equation

In [ ]:
# ── Source Term: Oscillating emitters at the two slits ──
# S(x,y,t) = S0 * Gaussian(x) * [Gaussian(y - d/2) + Gaussian(y + d/2)] * exp(-i*ω0*t)
S_expr = S0 * sp.exp(-((x - x0)/wx)**2) * \
         (sp.exp(-((y - d/2)/wy)**2) + sp.exp(-((y + d/2)/wy)**2)) * \
         sp.exp(-sp.I * omega0 * t)

# ∂ψ/∂t = psiOp(-i(ξ² + η²), ψ)  +  S(x,y,t)
#        ───────────────────────    ──────────
#        Linear diffraction (Fourier)  Huygens Source (Physical)

equation = sp.Eq(
    sp.diff(psi_field, t),
    psiOp(symbol_linear, psi_field) + S_expr
)

print("Linear Schrödinger Equation with Source:")
print("  ∂ψ/∂t = psiOp(-i(ξ² + η²), ψ) + S(x,y,t)")

## 5. Initial conditions: Vacuum state

In [ ]:
def initial_condition_schrodinger(xx, yy):
    """
    Start from a vacuum state (ψ = 0).
    The continuous source term will inject waves into the system over time.
    """
    return np.zeros_like(xx, dtype=np.complex128)

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Boundaries are far enough to avoid reflections
    initial_condition=initial_condition_schrodinger,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization
We visualize the amplitude $|\psi|$ using a 2D image plot (`imshow`), which is much better for resolving fine interference fringes than a 3D surface plot. The contour overlay will highlight the constructive and destructive interference bands.

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='abs',    # Show |ψ| (envelope amplitude)
    overlay='contour',  # Contours reveal the interference fringes
    mode='imshow',      # 2D raster is ideal for diffraction patterns
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='abs',    # Show |ψ| (envelope amplitude)
    overlay='contour',  # Contours reveal the interference fringes
    mode='surface',      # 2D raster is ideal for diffraction patterns
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
# ani.save('schrodinger_young_slits.mp4', writer='ffmpeg', fps=20, dpi=150)
# print("✅ Saved to schrodinger_young_slits.mp4")